# 06 · Ejercicios

Seis ejercicios para fijar lo de los cuadernos 01 a 05. Cada uno tiene un
enunciado, una pista y, al final del cuaderno, una **solución**. Intenta
resolverlos antes de bajar. El *bootstrap* deja cargadas las utilidades; no
hace falta el LLM para los ejercicios 1–4 y 6. El 5 sí propone tocar el *prompt*,
y su solución se puede leer aunque no generes de nuevo.


In [1]:
from pathlib import Path
import os, sys, json, time, gc, textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _raiz():
    aqui = Path.cwd().resolve()
    for cand in [aqui, *aqui.parents]:
        if (cand / "src" / "rag_utils.py").exists() and (cand / "notebooks").is_dir():
            return cand
    raise FileNotFoundError(
        "No encuentro la raiz del proyecto de RAG "
        "(src/rag_utils.py + notebooks/)."
    )

RAIZ = _raiz()
sys.path.insert(0, str(RAIZ / "src"))

from rag_utils import (
    preparar_rutas,
    fijar_semilla,
    dispositivo,
    configurar_proxy_y_hf,
    comprobar_entorno,
    probar_conexion_hf,
    texto_ayuda_proxy,
    modo_rapido,
    guardar_json,
    cargar_json,
    cargar_corpus,
    cargar_preguntas,
    fragmentar_documentos,
    resumen_fragmentos,
    Codificador,
    similitud_coseno,
    IndiceNumpy,
    IndiceFAISS,
    IndiceBM25,
    fusion_rrf,
    Reranker,
    evaluar_recuperacion,
    construir_mensajes,
    construir_contexto,
    GeneradorLocal,
    contiene_hechos,
    parece_abstencion,
    PipelineRAG,
    evaluar_fidelidad,
    MODELO_EMBEDDINGS_DEFECTO,
    MODELO_RERANKER_DEFECTO,
    MODELO_LLM_DEFECTO,
    MODELO_LLM_RESERVA,
)

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

plt.rcParams.update({
    "figure.figsize": (8, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "figure.dpi": 110,
})

RUTAS = preparar_rutas(RAIZ)
SEMILLA = fijar_semilla()
CFG_RED = configurar_proxy_y_hf(RAIZ, verbose=True)
DEVICE = dispositivo()
MODO_RAPIDO = modo_rapido(True)
print(f"Raiz:        {RAIZ}")
print(f"Dispositivo: {DEVICE}")
print(f"Semilla:     {SEMILLA}")
print(f"Modo rapido: {MODO_RAPIDO}")


Configuración de red y Hugging Face
  HTTP_PROXY definido:  False
  HTTPS_PROXY definido: False
  NO_PROXY definido:    False
  Certificado SSL (SSL_CERT_FILE / REQUESTS_CA_BUNDLE): False
  Token de Hugging Face presente: False (no se muestra el valor)
  HF_HOME: /workspace/rag/cache-hf
  Modo offline HF: False
  No hizo falta ajustar variables: ya estaban definidas.
Raiz:        /workspace/rag
Dispositivo: cpu
Semilla:     42
Modo rapido: True


In [2]:
# --- Mandos (comunes a los ejercicios) ---
MODELO_EMB = MODELO_EMBEDDINGS_DEFECTO
documentos = cargar_corpus(RUTAS["corpus"])
preguntas = cargar_preguntas(RUTAS["datos"] / "preguntas.json")
preguntas_hecho = [p for p in preguntas if p.get("tipo") != "no_esta"]
codificador = Codificador(MODELO_EMB, dispositivo_destino=str(DEVICE))
indice = IndiceFAISS.cargar(RUTAS["modelos"] / "indice_faiss")
print(f"docs={len(documentos)}  fragmentos={len(indice.fragmentos)}  preguntas hecho={len(preguntas_hecho)}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

docs=9  fragmentos=77  preguntas hecho=18


/workspace/rag/src/rag_utils.py:594: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = int(self.modelo.get_sentence_embedding_dimension())


## Ejercicio 1 — Tamaño de *chunk* y un hecho frágil

Elige un hecho corto y concreto, por ejemplo `22 días laborables` o
`ci-retencion.yml`. Trocea el corpus con `tamano=150` y `solape=0`, y otra vez
con `tamano=150` y `solape=60`. ¿En cuántos fragmentos aparece el hecho **entero**
en cada caso? ¿Qué implica eso para *recall* si la pregunta es precisamente
esa cifra?

**Pista:** `fragmentar_documentos` y una comprensión de listas sobre `f.texto`.


In [3]:
# Escribe tu código aquí (ejercicio 1)
HECHO = "22 días laborables"
# ...


## Ejercicio 2 — Añade un documento y reindexa

Inventa un documento Markdown breve (por ejemplo, la política de *tickets de
comida* de NubeAndina: 18 soles por día laborable, no acumulables) en
`datos/corpus/`, recárgalo, vuelve a trocear y a embeber, y formula una
pregunta nueva con su `docs_relevantes`. Comprueba que el denso o BM25 lo
recuperan en el top-3.

**Pista:** no hace falta tocar `preguntas.json` si construyes el diccionario de
la pregunta en el cuaderno. Al terminar, puedes borrar el archivo para dejar el
corpus como estaba.


In [4]:
# Escribe tu código aquí (ejercicio 2)
# ruta = RUTAS["corpus"] / "nubeandina-tickets-comida.md"
# ...


## Ejercicio 3 — *k* = 1 frente a *k* = 5

Toma la pregunta de las vacaciones de NubeAndina. Lista el top-5 denso. Si solo
le pasaras el primero al LLM, ¿viaja el hecho de los 22 días **y** el del cierre
andino? ¿Y con k=5? Comenta el riesgo de un *k* demasiado pequeño y el de uno
demasiado grande.


In [5]:
# Escribe tu código aquí (ejercicio 3)
pregunta = "¿Cuántos días laborables de vacaciones tiene un empleado de NubeAndina y qué es el cierre andino?"
# ...


## Ejercicio 4 — BM25 frente a denso en un identificador

Compara BM25 y la búsqueda densa en una pregunta que cite un identificador
literal (`gke-prod-andina-01` o `ci-retencion.yml`) y en una paráfrasis sin ese
token («¿cómo se llama el clúster donde NubeAndina sirve producción?»). ¿Quién
gana en cada una? ¿Qué haría la fusión RRF?

**Pista:** `IndiceBM25` sobre `indice.fragmentos` y `fusion_rrf`.


In [6]:
# Escribe tu código aquí (ejercicio 4)
# ...


## Ejercicio 5 — Un *prompt* más estricto

El *prompt* de `construir_mensajes` pide abstenerse si el dato no está. Redacta
una variante todavía más estricta (por ejemplo: «si no puedes citar un
identificador de fragmento, no respondas la cifra»). ¿En qué tipo de pregunta
ayudaría, y en cuál podría dar un falso «no lo encuentro» aunque el contexto
sí tenga el dato?

No es obligatorio ejecutar el LLM otra vez: razona con lo que viste en el
cuaderno 04.


In [7]:
# Opcional: reescribe el sistema y, si quieres, genera una pregunta no_esta.
# NUEVO_SISTEMA = """..."""


## Ejercicio 6 — Una caché de consultas

Implementa un diccionario que, dada la pregunta exactamente igual, reutilice
los fragmentos ya recuperados (y, si te animas, la respuesta). Mide el tiempo
de la primera llamada y de la segunda. ¿Qué clave usarías en producción, más
allá del texto crudo? (Pista: la versión del índice y del modelo de embeddings.)


In [8]:
# Escribe tu código aquí (ejercicio 6)
# cache = {}
# ...


---

# Soluciones

Baja hasta aquí cuando hayas intentado cada ejercicio. El código de las
soluciones es deliberadamente corto; no es «la» implementación de producción.


### Solución 1


In [9]:
def n_cubren(tamano, solape, hecho="22 días laborables"):
    frags = fragmentar_documentos(documentos, estrategia="solape" if solape else "fijo", tamano=tamano, solape=solape)
    hits = [f for f in frags if hecho.lower() in f.texto.lower()]
    print(f"tamano={tamano} solape={solape}: {len(hits)} fragmentos cubren el hecho (total {len(frags)})")
    for h in hits:
        print("  ", h.id)
    return hits

n_cubren(150, 0)
n_cubren(150, 60)
print(
    "Sin solape, un hecho corto cerca del borde de la ventana puede partirse "
    "y desaparecer de todos los fragmentos. El recall de esa pregunta se cae "
    "aunque el documento 'correcto' esté en el corpus."
)


tamano=150 solape=0: 1 fragmentos cubren el hecho (total 169)
   nubeandina-vacaciones-y-trabajo::#2
tamano=150 solape=60: 2 fragmentos cubren el hecho (total 276)
   nubeandina-vacaciones-y-trabajo::#3
   nubeandina-vacaciones-y-trabajo::#4
Sin solape, un hecho corto cerca del borde de la ventana puede partirse y desaparecer de todos los fragmentos. El recall de esa pregunta se cae aunque el documento 'correcto' esté en el corpus.


### Solución 2


In [10]:
ruta_extra = RUTAS["corpus"] / "nubeandina-tickets-comida.md"
ruta_extra.write_text(
    "# NubeAndina: tickets de comida\n\n"
    "Cada empleado de jornada completa recibe un ticket de comida de **18 soles** "
    "por día laborable presencial. Los tickets no son acumulables ni canjeables "
    "por dinero. El día de trabajo remoto no genera ticket. Esta política es "
    "ficticia y solo existe para el laboratorio de RAG.\n",
    encoding="utf-8",
)
docs2 = cargar_corpus(RUTAS["corpus"])
frags2 = fragmentar_documentos(docs2, estrategia="solape", tamano=400, solape=80)
vec2 = codificador.codificar([f.texto for f in frags2], progreso=False)
ind2 = IndiceFAISS(vec2, frags2)
preg = "¿De cuántos soles es el ticket de comida de NubeAndina en un día presencial?"
qv = codificador.codificar([preg], progreso=False)[0]
hits = ind2.buscar(qv, k=3)
print("Top-3:")
for r in hits:
    print(f"  {r.rango}. {r.fragmento.doc_id}  {r.puntuacion:.3f}")
print("¿Aparece el documento nuevo?", any(r.fragmento.doc_id == "nubeandina-tickets-comida" for r in hits))
ruta_extra.unlink()
print("Documento temporal borrado; el corpus queda como al principio.")


Top-3:
  1. nubeandina-tickets-comida  0.633
  2. nubeandina-vacaciones-y-trabajo  0.499
  3. nubeandina-vacaciones-y-trabajo  0.485
¿Aparece el documento nuevo? True
Documento temporal borrado; el corpus queda como al principio.


### Solución 3


In [11]:
pregunta = "¿Cuántos días laborables de vacaciones tiene un empleado de NubeAndina y qué es el cierre andino?"
qv = codificador.codificar([pregunta], progreso=False)[0]
hits = indice.buscar(qv, k=5)
for r in hits:
    t = r.fragmento.texto.lower()
    tiene_22 = "22" in t and "vacacion" in t.replace("ó", "o")
    tiene_cierre = "cierre andino" in t
    print(f"{r.rango}. {r.fragmento.id}  22dias={tiene_22}  cierre={tiene_cierre}")
print(
    "Con k=1 solo viaja el primer fragmento: si el cierre andino quedó en el "
    "siguiente trozo, el LLM no lo verá. Con k=5 es más probable que viajen "
    "ambos hechos, a costa de más ruido y más tokens."
)


1. nubeandina-vacaciones-y-trabajo::#1  22dias=True  cierre=True
2. nubeandina-vacaciones-y-trabajo::#3  22dias=False  cierre=True
3. nubeandina-vacaciones-y-trabajo::#0  22dias=True  cierre=False
4. nubeandina-vacaciones-y-trabajo::#7  22dias=False  cierre=False
5. nubeandina-vacaciones-y-trabajo::#6  22dias=False  cierre=False
Con k=1 solo viaja el primer fragmento: si el cierre andino quedó en el siguiente trozo, el LLM no lo verá. Con k=5 es más probable que viajen ambos hechos, a costa de más ruido y más tokens.


### Solución 4


In [12]:
bm25 = IndiceBM25(indice.fragmentos)

def ver(pregunta):
    qv = codificador.codificar([pregunta], progreso=False)[0]
    denso = indice.buscar(qv, k=3)
    lex = bm25.buscar(pregunta, k=3)
    hib = fusion_rrf([denso, lex], k=3)
    print("=" * 60)
    print(pregunta)
    for nombre, lista in ("BM25", lex), ("denso", denso), ("hibrido", hib):
        print(f"  {nombre}:", ", ".join(r.fragmento.doc_id for r in lista))

ver("¿Qué es gke-prod-andina-01?")
ver("¿Cómo se llama el clúster donde NubeAndina sirve producción?")
print(
    "El identificador literal favorece a BM25. La parafrasis favorece a la busqueda "
    "densa. RRF no 'promedia puntuaciones'; suma 1/(k+rango) y por eso puede "
    "conservar el acierto de cualquiera de las dos listas."
)


¿Qué es gke-prod-andina-01?
  BM25: nubeandina-empresa, nubeandina-datos-y-seguridad, nubeandina-datos-y-seguridad
  denso: nubeandina-datos-y-seguridad, curso-reglamento, curso-avance-3
  hibrido: nubeandina-datos-y-seguridad, nubeandina-empresa, curso-reglamento
¿Cómo se llama el clúster donde NubeAndina sirve producción?
  BM25: nubeandina-empresa, nubeandina-datos-y-seguridad, nubeandina-empresa
  denso: nubeandina-empresa, nubeandina-empresa, nubeandina-empresa
  hibrido: nubeandina-empresa, nubeandina-empresa, nubeandina-datos-y-seguridad
El identificador literal favorece a BM25. La parafrasis favorece a la busqueda densa. RRF no 'promedia puntuaciones'; suma 1/(k+rango) y por eso puede conservar el acierto de cualquiera de las dos listas.


### Solución 5

Una variante más estricta del sistema podría decir:

> Responde únicamente si puedes apoyar cada cifra o nombre en una cita
> `[fragmento-id]`. Si no puedes citar, escribe exactamente
> «No encuentro esa información en los documentos.» No completes con conocimiento
> previo.

Ayudaría en las preguntas *no_esta* (teléfono, sueldo) y cuando el recuperador
trae documentos del tema pero **sin** la cifra pedida. El riesgo es el falso
negativo: un modelo pequeño, aunque tenga el hecho en el contexto, a veces no
genera la cita y, si obedece al pie de la letra, se abstiene. Por eso se miden
*a la vez* la tasa de abstención y la cobertura de hechos, no solo una de las dos.

En producción, este guardarraíl se combina con filtros (¿el fragmento es del
tenant correcto?) y con trazas (¿qué se le mostró realmente al modelo?).


### Solución 6


In [13]:
class CacheRecuperacion:
    def __init__(self, pipeline, version_indice="indice_faiss_v1"):
        self.pipeline = pipeline
        self.version = version_indice
        self.mem = {}

    def clave(self, pregunta, k):
        return (self.version, self.pipeline.codificador.nombre, k, pregunta.strip())

    def recuperar(self, pregunta, k=4):
        c = self.clave(pregunta, k)
        if c in self.mem:
            return self.mem[c], True
        hits = self.pipeline.recuperar(pregunta, k=k)
        self.mem[c] = hits
        return hits, False

pipe = PipelineRAG(codificador=codificador, indice_denso=indice, metodo="denso", k=4)
cache = CacheRecuperacion(pipe)
preg = "¿Cuál es el tracking URI por defecto de MLflow en el laboratorio?"
t0 = time.perf_counter()
_, hit1 = cache.recuperar(preg, k=4)
t1 = time.perf_counter() - t0
t0 = time.perf_counter()
_, hit2 = cache.recuperar(preg, k=4)
t2 = time.perf_counter() - t0
print(f"primera (miss={not hit1}): {t1:.4f}s   segunda (hit={hit2}): {t2:.4f}s")
print("Entradas en cache:", len(cache.mem))
print(
    "En produccion la clave deberia incluir la version del indice, el nombre "
    "(y hash) del modelo de embeddings, el k y una normalizacion de la pregunta. "
    "Si el reglamento cambia, sube la version y la cache antigua deja de usarse."
)


primera (miss=True): 0.0095s   segunda (hit=True): 0.0000s
Entradas en cache: 1
En produccion la clave deberia incluir la version del indice, el nombre (y hash) del modelo de embeddings, el k y una normalizacion de la pregunta. Si el reglamento cambia, sube la version y la cache antigua deja de usarse.


## Cierre de la serie

Has recorrido el camino completo: embeddings, troceado, índice, recuperación
(léxica, densa e híbrida), generación con y sin contexto, y las notas para
llevarlo a un servicio. El siguiente paso, en el marco del curso, es tratar
este *pipeline* como cualquier otro artefacto: medirlo, versionarlo y, cuando
toque, empaquetarlo.
